# Common NLP tasks and the improved Marvin bot

Execute this notebook from its own folder after the shared data setup.
The assignment implementation is [bot.py](bot.py). Its rules handle *why*
and *how* as whole words, use TextBlob sentiment and a ConLL noun-phrase
extractor, pluralize the detected topic, and retain one topic across turns.

In [1]:
from collections import Counter
from pathlib import Path
import nltk
import pandas as pd
from IPython.display import display
from nltk import bigrams, pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob, Word
from bot import reply, EXTRACTOR

nltk.data.path.insert(0, str(Path('../data/nltk_data').resolve()))
sentence = 'The quick red fox jumped over the lazy brown dog.'
tokens = word_tokenize(sentence)
display(pd.DataFrame(pos_tag(tokens), columns=['token', 'part_of_speech']))
words = [word.casefold() for word in tokens if word.isalpha()]
print('Frequencies:', Counter(words))
print('Bigrams:', list(bigrams(words)))
assert Counter(words)['the'] == 2
assert list(bigrams(words))[0] == ('the', 'quick')

,token,part_of_speech
0,The,DT
1,quick,JJ
2,red,JJ
3,fox,NN
4,jumped,VBD
5,over,IN
6,the,DT
7,lazy,JJ
8,brown,NN
9,dog,NN


Frequencies: Counter({'the': 2, 'quick': 1, 'red': 1, 'fox': 1, 'jumped': 1, 'over': 1, 'lazy': 1, 'brown': 1, 'dog': 1})
Bigrams: [('the', 'quick'), ('quick', 'red'), ('red', 'fox'), ('fox', 'jumped'), ('jumped', 'over'), ('over', 'the'), ('the', 'lazy'), ('lazy', 'brown'), ('brown', 'dog')]


In [2]:
blob = TextBlob('I went for a walk and saw a lovely cat.', np_extractor=EXTRACTOR)
print('Noun phrases:', blob.noun_phrases)
print('Plural:', Word('mouse').pluralize())
print('Verb lemma:', WordNetLemmatizer().lemmatize('flew', pos='v'))
print('Sentiment:', blob.sentiment)
assert 'lovely cat' in blob.noun_phrases
assert WordNetLemmatizer().lemmatize('flew', pos='v') == 'fly'

Noun phrases: ['lovely cat']
Plural: mice


Verb lemma: fly
Sentiment: Sentiment(polarity=0.5, subjectivity=0.75)


## Scripted assignment and challenge test

These are supplied test messages, not a conversation with a friend. A fixed
topic is a limited memory mechanism: it can carry a cat into a follow-up,
but cannot resolve arbitrary pronouns or distinguish several entities.

In [3]:
memory = {}
messages = [
    'I saw a lovely cat.', 'It was wonderful!', 'Why do cats purr?',
    'How can I learn?', 'Somehow I feel good.',
    'This is awful and horrible.', '', ' BYE ',
]
rows = []
for message in messages:
    response = reply(message, memory)
    rows.append({'input': message, 'response': response, 'remembered_topic': memory.get('topic')})
display(pd.DataFrame(rows))
assert 'lovely cats' in rows[1]['response']
assert 'cause' in rows[2]['response']
assert 'step' in rows[3]['response']
assert 'step' not in rows[4]['response']
assert rows[-1]['response'] is None

,input,response,remembered_topic
0,I saw a lovely cat.,That sounds positive. Can you tell me more abo...,lovely cats
1,It was wonderful!,That sounds very positive. Can you tell me mor...,lovely cats
2,Why do cats purr?,What cause do you think explains it? Can you t...,lovely cats
3,How can I learn?,What first step could you try? Can you tell me...,lovely cats
4,Somehow I feel good.,That sounds very positive. Can you tell me mor...,lovely cats
5,This is awful and horrible.,"Oh dear, that sounds bad. Can you tell me more...",lovely cats
6,,"Please type a message, or 'bye' to stop.",lovely cats
7,BYE,None,lovely cats


## Knowledge checks and further features

Tokenization and part-of-speech tags expose structure; noun phrases identify
candidate topics. Lemmatization needs a part of speech: *flew* is a form of
*fly* as a verb. Embeddings instead represent tokens by vectors learned from
context, allowing similarity beyond exact word matching.

The bot combines a trained chunker with a lexicon sentiment estimator and
manual question rules. Its apparent sympathy does not demonstrate
understanding. A labelled intent classifier could replace the growing rule
list, with a low-confidence fallback and held-out evaluation.

The next lesson's feature-extraction challenge is also implemented here:
question words, noun phrases, polarity, and the previous topic all influence
the answer. Blank input and a case-insensitive, whitespace-trimmed `bye`
are handled explicitly. No human deception experiment was performed.